# Rotation Orientation Sweep -- VectorClassification OCR angle diagnostic

Empirically tests the sign/offset convention in `P3_Vector_Parsing/VectorClassification/parse.py`'s
`quad -> Text.direction` composition (`_quad_rotation_deg` + `_normalize_rotation` +
`transform_direction`) by rotating one real hand-labelled vector text box through a full sweep of
orientations and running each through the real `PaddleDetectBackend`/`PaddleRecBackend`.

**Inputs**: a PDF and a `vector_labels.json` produced by `scripts/label/vector_label.py`
(a single labelled text box is enough -- pick its index below).

**What this checks**: if the composition is correct, the *recovered* rotation delta (relative to
the unrotated baseline) should track the *applied* rotation delta 1:1 (`y = x`) across the whole
sweep. A `y = -x` relationship (or any other systematic offset) confirms and quantifies the sign
error traced analytically in the audit.

First run downloads PaddleOCR models if not already cached (same as any other real-OCR use in
this repo).

In [ ]:
from __future__ import annotations

import math

import matplotlib.pyplot as plt
import numpy as np

from rastervec.P1_Reading_Native.reader import Reader
from rastervec.P1_Reading_Native.vector_extract import extract_vectors
from rastervec.Evaluation.Labelling.label_schema import load_labels, path_signature
from rastervec.commons.helpers.geometry import (
    compute_origin, transform_direction, transform_point, transform_vector, union_bbox,
)
from rastervec.commons.models import Text
from rastervec.commons.renderer import ocr_prep, pixel_to_page_bbox
from rastervec.P3_Vector_Parsing.VectorClassification.config import (
    MAX_RENDER_DPI, MIN_RENDER_SIDE_PX, OCR_DPI,
)
from rastervec.P3_Vector_Parsing.VectorClassification.paddle_engine import (
    PaddleDetectBackend, PaddleRecBackend, _normalize_bgr, _normalize_rotation,
    _quad_rotation_deg, _rotate_crop,
)
from rastervec.P3_Vector_Parsing.VectorClassification.parse import _cluster_render_padding

## 1. Config -- set these

In [ ]:
PDF_PATH = "path/to/your.pdf"                 # <-- set this
LABELS_PATH = "path/to/vector_labels.json"    # <-- set this (vector_label.py's --out file)
PAGE_INDEX = 0                                # <-- the page the label is on

reader = Reader(PDF_PATH)
page = reader.get_page(PAGE_INDEX)
labels = load_labels(LABELS_PATH)
print(f"loaded {len(labels.entries)} label entries from {LABELS_PATH}")

## 2. Pick one vector-source label

In [ ]:
vector_entries = [e for e in labels.entries if e.source == "vector" and e.page_index == PAGE_INDEX]
for i, e in enumerate(vector_entries):
    print(f"[{i}] text={e.text!r} bbox={e.cluster_bbox} expected_rotation={e.expected_rotation}")

LABEL_INDEX = 0  # <-- set this
entry = vector_entries[LABEL_INDEX]
print("using:", entry.text, entry.cluster_bbox)

## 3. Resolve the label's real backing vectors

In [ ]:
all_vectors = extract_vectors(page)
sig_set = set(entry.vector_signatures)
group_vectors = [v for v in all_vectors if path_signature(v) in sig_set]
assert group_vectors, "no vectors matched this label's vector_signatures -- re-extract or re-label"
print(f"resolved {len(group_vectors)} vector(s) backing this label")

## 4. Rotate the label's own geometry about its own bbox center

Uses the exact functions the audit flagged (`transform_point`/`transform_vector`), so this sweep
exercises the real production code, not a reimplementation.

In [ ]:
def rotate_vectors_about_center(vectors, angle_deg):
    x0, y0, x1, y1 = union_bbox([v.bbox for v in vectors])
    center = ((x0 + x1) / 2.0, (y0 + y1) / 2.0)
    rotated_center = transform_point(center, offset=(0.0, 0.0), rotation_deg=angle_deg)
    offset = (center[0] - rotated_center[0], center[1] - rotated_center[1])
    return [transform_vector(v, offset=offset, rotation_deg=angle_deg) for v in vectors]


ANGLES = list(range(0, 360, 15))  # "all possible orientations" -- one sample every 15 degrees

## 5. Sweep: rotate -> render -> real PaddleOCR detect+recognize -> recover angle

Mirrors `parse.py`'s own inner loop exactly (same padding helper, same backends, same
`_quad_rotation_deg`/`_normalize_rotation`/`transform_direction` composition) -- just run once per
swept angle instead of once per pipeline cluster.

In [ ]:
det_backend = PaddleDetectBackend()
rec_backend = PaddleRecBackend()

results = []  # each: dict(angle, image, bgr, quad, ocr_text, recovered_angle)

for angle in ANGLES:
    rotated = rotate_vectors_about_center(group_vectors, angle)
    padding = _cluster_render_padding(rotated)
    image, dpi_used = ocr_prep.render_cluster_with_dynamic_dpi(
        rotated, OCR_DPI, MIN_RENDER_SIDE_PX, MAX_RENDER_DPI, padding,
    )
    bgr = _normalize_bgr(np.asarray(image))
    quads = det_backend.detect(bgr)
    if not quads:
        print(f"angle={angle}: no detection, skipping")
        continue
    quad = quads[0]
    crop = _rotate_crop(bgr, quad)[:, :, ::-1]
    box = rec_backend.recognize_crops([crop])[0]

    rotate_deg = _normalize_rotation(_quad_rotation_deg(quad) + box.flip_deg)
    direction = transform_direction((1.0, 0.0), rotate_deg)
    bbox = pixel_to_page_bbox(rotated, dpi_used, quad.tolist(), padding)
    text = Text(
        text=box.text, bbox=bbox, direction=direction, origin=compute_origin(bbox, direction),
        font="", font_size=0.0, color=None, flags=0, ascender=None, descender=None, wmode=0,
        block_no=0, line_no=0, word_no=0, page_index=page.meta.index,
        seqno=min(v.seqno for v in rotated), confidence=box.confidence,
        source="ocr", orientation_source="ocr",
    )
    results.append(dict(
        angle=angle, image=image, bgr=bgr, quad=quad, ocr_text=box.text,
        recovered_angle=text.angle(),
    ))
    print(f"angle={angle:>4}  ocr_text={box.text!r:>20}  recovered_angle={text.angle():.1f}")

## 6. Compare applied vs. recovered rotation

If the composition is correct, `recovered_delta` should equal `applied_delta` for every angle
(a straight `y = x` line below). A `y = -x` line (or any other consistent relationship) confirms
and quantifies a sign/offset bug.

In [ ]:
assert results, "no angle produced a detection -- can't diagnose, check the label/PDF"

baseline_angle = results[0]["recovered_angle"]  # results[0] is ANGLES[0], normally 0
applied_deltas = []
recovered_deltas = []
for r in results:
    applied_deltas.append(_normalize_rotation(r["angle"]))
    recovered_deltas.append(_normalize_rotation(r["recovered_angle"] - baseline_angle))

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([-90, 90], [-90, 90], "k--", alpha=0.3, label="y = x (correct)")
ax.plot([-90, 90], [90, -90], "r--", alpha=0.3, label="y = -x (sign-flipped)")
ax.scatter(applied_deltas, recovered_deltas)
for a, r, res in zip(applied_deltas, recovered_deltas, results):
    ax.annotate(str(res["angle"]), (a, r), fontsize=7)
ax.set_xlabel("applied rotation delta (deg, wrapped to [-90,90))")
ax.set_ylabel("recovered rotation delta (deg, wrapped to [-90,90))")
ax.set_title("VectorClassification OCR rotation: applied vs. recovered")
ax.legend()
ax.set_aspect("equal")
plt.show()

## 7. Visual cross-check: rendered crop per angle

In [ ]:
n = len(results)
cols = 4
rows = math.ceil(n / cols)
fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
axes = np.atleast_1d(axes).ravel()
for ax, r in zip(axes, results):
    ax.imshow(r["image"])
    quad = r["quad"]
    poly = np.vstack([quad, quad[0]])
    ax.plot(poly[:, 0], poly[:, 1], "r-")
    ax.set_title(
        f"applied={r['angle']}  ocr={r['ocr_text']!r}\nrecovered={r['recovered_angle']:.1f}",
        fontsize=8,
    )
    ax.axis("off")
for ax in axes[n:]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 8. Conclusion

Fill in after running: does the plot in step 6 follow `y = x` (composition is correct) or
`y = -x` / another pattern (composition has a sign/offset bug)? If it's `y = -x`, the fix is to
remove (or flip the sign of) the `-theta` negation in
`P3_Vector_Parsing/VectorClassification/paddle_engine.py::_quad_rotation_deg` -- then re-run
`tests/rastervec/P3_Vector_Parsing/VectorClassification/test_parse.py` and
`test_paddle_engine.py` to confirm no regression.